In [1]:
import pandas as pd
import os
os.getcwd()

'/home/lixiangyu/zr/Annotate/ANNOTATE_new/10_downstream/SEGs/FPKM'

In [2]:
def compute_gene_length(gtf_file, output_file):
    print("读取 GTF...")

    cols = ["chr","source","feature","start","end","score","strand","frame","attr"]
    df = pd.read_csv(gtf_file, sep="\t", comment="#", names=cols)

    # 只保留 exon
    df = df[df["feature"] == "exon"].copy()

    # 提取 gene name
    df["gene"] = df["attr"].str.extract('gene_name "([^"]+)"')

    df = df.dropna(subset=["gene"])

    print("计算 union exon length...")

    gene_lengths = {}

    for gene, group in df.groupby("gene"):
        intervals = group[["start","end"]].values
        intervals = sorted(intervals, key=lambda x: x[0])

        merged = []
        for start, end in intervals:
            if not merged or start > merged[-1][1]:
                merged.append([start, end])
            else:
                merged[-1][1] = max(merged[-1][1], end)

        length = sum(e - s + 1 for s, e in merged)
        gene_lengths[gene] = length

    out = pd.DataFrame(list(gene_lengths.items()), columns=["gene","length"])
    out.to_csv(output_file, sep="\t", index=False)

    print(f"完成！输出：{output_file}")


# 用法
compute_gene_length("Homo_sapiens.GRCh38.115.gtf", "human_gene_length.tsv")
compute_gene_length("Mus_musculus.GRCm39.115.gtf", "mouse_gene_length.tsv")

读取 GTF...


/tmp/ipykernel_1154464/2058092916.py:5: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(gtf_file, sep="\t", comment="#", names=cols)


计算 union exon length...
完成！输出：human_gene_length.tsv
读取 GTF...


/tmp/ipykernel_1154464/2058092916.py:5: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(gtf_file, sep="\t", comment="#", names=cols)


计算 union exon length...
完成！输出：mouse_gene_length.tsv
